<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright 2026 Evan Gress -->

# 01 · Getting started with `fpc`

A notebook mixes code, results, plots, and explanation in one document — ideal for exploring a problem and showing your working.

**Run a cell:** Shift+Enter. Cells share variables, so run them top to bottom.

Start Jupyter from the repo root:
```bash
uv sync --group notebooks
uv run jupyter lab
```

## 1. Import a class and call a method
Every method takes **keyword arguments**, so the call says what each number means.

In [ ]:
from fpc import Dynamics, Thermo

dyn = Dynamics()
dyn.displacement_constant_acceleration(
    initial_position=0.0, initial_velocity=5.0, acceleration=9.81, time=3.0,
)

The last expression in a cell is displayed automatically. `help()` (or Shift+Tab in Jupyter) shows the docstring, including the units:

In [ ]:
help(Thermo.carnot_efficiency)

## 2. Many cases: lists of dicts and `**` unpacking

In [ ]:
launches = [{"speed": 20, "angle_deg": a} for a in (15, 30, 45, 60, 75)]
for case in launches:
    print(f"{case['angle_deg']:>3} deg -> {dyn.projectile_range(**case):6.2f} m")

## 3. Plot a sweep
A plot shows the whole trend — here, why 45° is the best launch angle without drag.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

angles = np.linspace(1, 89, 89)
ranges = [dyn.projectile_range(speed=20, angle_deg=a) for a in angles]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(angles, ranges)
ax.set_xlabel("Launch angle [deg]")
ax.set_ylabel("Range [m]")
ax.grid(alpha=0.3)
angles[int(np.argmax(ranges))]

## 4. Drag changes the answer
`simulate_projectile_drag` integrates the equations of motion with SciPy. Compare its range with the no-drag formula.

In [ ]:
# A tennis ball: 57 g, 6.7 cm diameter, Cd ~ 0.5
import math

t, x, y = dyn.simulate_projectile_drag(
    speed=20, angle_deg=45, mass=0.057, drag_coefficient=0.5,
    frontal_area=math.pi * 0.067**2 / 4,
)
no_drag = dyn.projectile_range(speed=20, angle_deg=45)
print(f"without drag: {no_drag:.1f} m   with drag: {x[-1]:.1f} m")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, y, label="with drag")
xs = np.linspace(0, no_drag, 100)
ax.plot(xs, xs - 9.81 * xs**2 / 20**2, "--", label="no drag")  # y = x*tan(45) - g*x^2/(2 v^2 cos^2 45)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.legend()
ax.grid(alpha=0.3)

## Next
- `02_pipe_flow_and_data.ipynb` — pandas tables, interpolation, root finding
- `03_beam_design.ipynb` — units with Pint, sizing and optimization
- `examples/` — complete worked problems as scripts